In [2]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../data/processed/enriched_events.parquet')
print(df.shape)

(19150868, 18)


In [3]:
float_cols = ['danceability', 'energy', 'valence', 'tempo', 'acousticness', 'loudness', 'popularity']
for col in float_cols:
    df[col] = df[col].astype('float32')

df['is_skip'] = df['is_skip'].astype('bool')
df['is_new_session'] = df['is_new_session'].astype('int8')
df['session_id'] = df['session_id'].astype('int32')

print(df.memory_usage(deep=True).sum() / 1e9, "GB")

6.19513532 GB


In [4]:
df = df.drop(columns=['track_name', 'match_type'], errors='ignore')

In [5]:
df.sort_values(['user_id', 'session_id', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)

In [6]:
df['session_position'] = df.groupby(['user_id', 'session_id']).cumcount() + 1

In [7]:
group_key = ['user_id', 'session_id']

df['prev_is_skip'] = df.groupby(group_key)['is_skip'].shift(1)
df['prev_is_skip'] = df['prev_is_skip'].fillna(-1)

for feat in ['energy', 'valence', 'danceability']:
    df[f'prev_{feat}'] = df.groupby(group_key)[feat].shift(1)
    df[f'{feat}_delta'] = df[feat] - df[f'prev_{feat}']

In [8]:
shifted = df.groupby(group_key)['is_skip'].shift(1)

df['rolling_skip_rate_5'] = (
    shifted.groupby([df['user_id'], df['session_id']])
    .rolling(window=5, min_periods=1)
    .mean()
    .reset_index(level=[0, 1], drop=True)
)

df['rolling_skip_rate_5'] = df['rolling_skip_rate_5'].fillna(-1)

In [9]:
df['prev_artist'] = df.groupby(group_key)['artist_name'].shift(1)
df['artist_repeat'] = (df['artist_name'] == df['prev_artist']).astype(int)
df.loc[df['prev_artist'].isna(), 'artist_repeat'] = -1

df['prev_genre'] = df.groupby(group_key)['track_genre'].shift(1)
df['genre_changed'] = np.where(
    df['track_genre'].isna() | df['prev_genre'].isna(),
    -1,
    (df['track_genre'] != df['prev_genre']).astype(int)
)

In [10]:
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

In [11]:
session_lengths = df.groupby(group_key)['session_position'].transform('max')
df['session_progress'] = df['session_position'] / session_lengths

In [12]:
df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)

shifted_skip = df.groupby('user_id')['is_skip'].shift(1)
df['user_historical_skip_rate'] = (
    shifted_skip.groupby(df['user_id']).expanding().mean().reset_index(level=0, drop=True)
)
df['user_historical_skip_rate'] = df['user_historical_skip_rate'].fillna(-1)

df = df.sort_values(['user_id', 'session_id', 'timestamp']).reset_index(drop=True)

In [13]:
for feat in ['energy', 'valence']:
    shifted_feat = df.groupby(group_key)[feat].shift(1)
    df[f'rolling_{feat}_5'] = (
        shifted_feat.groupby([df['user_id'], df['session_id']])
        .rolling(window=5, min_periods=1)
        .mean()
        .reset_index(level=[0, 1], drop=True)
    )
    df[f'rolling_{feat}_5'] = df[f'rolling_{feat}_5'].fillna(-1)

In [14]:
feature_cols = [
    'session_position', 'session_progress', 'prev_is_skip', 'rolling_skip_rate_5',
    'artist_repeat', 'genre_changed', 'hour_sin', 'hour_cos', 'day_of_week',
    'user_historical_skip_rate', 'rolling_energy_5', 'rolling_valence_5',
    'energy_delta', 'valence_delta', 'danceability_delta'
]
print(df[feature_cols].describe())

       session_position  session_progress  rolling_skip_rate_5  artist_repeat  \
count      1.915087e+07      1.915087e+07         1.915087e+07   1.915087e+07   
mean       5.808229e+01      5.272020e-01        -4.632751e-02   4.049159e-01   
std        1.794380e+02      2.915442e-01         2.387926e-01   5.914110e-01   
min        1.000000e+00      1.865672e-04        -1.000000e+00  -1.000000e+00   
25%        6.000000e+00      2.758621e-01         0.000000e+00   0.000000e+00   
50%        1.700000e+01      5.241935e-01         0.000000e+00   0.000000e+00   
75%        4.500000e+01      7.777778e-01         0.000000e+00   1.000000e+00   
max        5.360000e+03      1.000000e+00         1.000000e+00   1.000000e+00   

       genre_changed      hour_sin      hour_cos   day_of_week  \
count   1.915087e+07  1.915087e+07  1.915087e+07  1.915087e+07   
mean   -9.751041e-01 -1.922429e-01  2.084515e-03  2.948802e+00   
std     1.973759e-01  6.844663e-01  7.032384e-01  1.996454e+00   
min   

In [16]:
# Cast is_skip to int8 (0/1) first, so prev_is_skip stays a clean numeric column
# (mixing bool True/False with the -1 sentinel was creating an ambiguous 'object' dtype)
df['prev_is_skip'] = df.groupby(group_key)['is_skip'].shift(1)
df['prev_is_skip'] = df['prev_is_skip'].fillna(-1).astype('int8')

print(df['prev_is_skip'].dtype)
print(df['prev_is_skip'].value_counts())

int8
prev_is_skip
 0    17953635
-1     1041883
 1      155350
Name: count, dtype: int64


In [17]:
df.to_parquet('../data/processed/feature_engineered.parquet', index=False)
print("Saved:", df.shape)

Saved: (19150868, 37)
